# Advanced 09 — Agent Identity Security Posture Management & Threat Defense

**Enterprise scenario:** a claims-agent ecosystem contains an orchestrator, research agent, data agent, MCP tools, vector database, claims API, KMS signing service, cloud roles and an external partner. You will compromise identities, enumerate attack paths, detect abuse, contain it and measure residual posture.

> The lab uses safe local simulations. It does not attack external systems.


In [ ]:
import networkx as nx, pandas as pd, json, math, statistics
from datetime import datetime,timedelta,timezone
NOW=datetime.now(timezone.utc)


## Lab 1 — Build the identity graph

In [ ]:
G=nx.DiGraph()
nodes={"user:alice":"human","agent:orchestrator":"agent","agent:research":"agent","agent:data":"agent",
"tool:vector":"tool","api:claims":"api","kms:claims":"kms","role:claims":"cloud-role","external:vendor":"external-agent"}
for n,k in nodes.items(): G.add_node(n,kind=k)
edges=[("user:alice","agent:orchestrator","sponsors"),("agent:orchestrator","agent:research","delegates"),
("agent:orchestrator","agent:data","delegates"),("agent:research","tool:vector","can_access"),
("agent:data","api:claims","can_access"),("api:claims","role:claims","runs_as"),
("role:claims","kms:claims","can_sign"),("external:vendor","agent:research","federated_access")]
for a,b,r in edges:G.add_edge(a,b,relation=r)
list(G.edges(data=True))

## Lab 2 — Tag crown jewels

In [ ]:
crown_jewels={"kms:claims","api:claims"}
nx.set_node_attributes(G,{n:(n in crown_jewels) for n in G.nodes},"crown_jewel")
[n for n,d in G.nodes(data=True) if d["crown_jewel"]]

## Lab 3 — Enumerate attack paths

In [ ]:
list(nx.all_simple_paths(G,"external:vendor","kms:claims",cutoff=8))

## Lab 4 — Calculate blast radius

In [ ]:
sorted(nx.descendants(G,"agent:orchestrator"))

## Lab 5 — Find choke points

In [ ]:
sorted(nx.betweenness_centrality(G).items(),key=lambda x:x[1],reverse=True)[:5]

## Lab 6 — Detect toxic privilege combinations

In [ ]:
privs={"agent:orchestrator":{"create_subagent","delegate"},
"role:claims":{"read_secrets","sign"},"ci:deploy":{"deploy","write_config"}}
toxic=[("create_subagent","delegate"),("read_secrets","deploy"),("write_policy","approve_policy")]
[(p,pair) for p,ps in privs.items() for pair in toxic if set(pair)<=ps]

## Lab 7 — Credential risk score

In [ ]:
credentials=pd.DataFrame([
{"id":"tok-a","lifetime_min":10,"scoped":1,"sender_bound":1,"leaked":0},
{"id":"key-b","lifetime_min":525600,"scoped":0,"sender_bound":0,"leaked":0},
{"id":"tok-c","lifetime_min":60,"scoped":1,"sender_bound":0,"leaked":1}])
credentials["risk"]=credentials.apply(lambda r:min(100,(r.lifetime_min/1440)*20+(1-r.scoped)*25+(1-r.sender_bound)*20+r.leaked*100),axis=1)
credentials

## Lab 8 — Find long-lived credentials

In [ ]:
credentials[credentials.lifetime_min>1440]

## Lab 9 — Model delegation

In [ ]:
delegations=pd.DataFrame([
{"parent":"agent:orchestrator","child":"agent:research","parent_scope":{"read"},"child_scope":{"read"},"minutes":20,"depth":1},
{"parent":"agent:research","child":"agent:rogue","parent_scope":{"read"},"child_scope":{"read","write"},"minutes":120,"depth":2}])
delegations

## Lab 10 — Detect over-delegation

In [ ]:
delegations["escalated"]=delegations.apply(lambda r:not r.child_scope.issubset(r.parent_scope),axis=1)
delegations[delegations.escalated]

## Lab 11 — Detect excessive delegation depth

In [ ]:
delegations[delegations.depth>1]

## Lab 12 — Tool-mediated lateral movement

In [ ]:
G.add_edge("tool:vector","api:claims",relation="misconfigured_pivot")
list(nx.all_simple_paths(G,"agent:research","api:claims",cutoff=4))

## Lab 13 — Environment isolation

In [ ]:
events=pd.DataFrame([{"principal":"agent:dev","principal_env":"dev","resource":"claims","resource_env":"prod"},
{"principal":"agent:prod","principal_env":"prod","resource":"claims","resource_env":"prod"}])
events["cross_env"]=events.principal_env!=events.resource_env
events

## Lab 14 — Audience anomaly

In [ ]:
tokens=[{"id":"t1","aud":"claims-api","used_at":"claims-api"},{"id":"t2","aud":"vector-api","used_at":"claims-api"}]
[x for x in tokens if x["aud"]!=x["used_at"]]

## Lab 15 — Replay cache

In [ ]:
seen=set()
def replay_check(jti):
    replay=jti in seen; seen.add(jti); return replay
replay_check("proof-1"),replay_check("proof-1")

## Lab 16 — Revoked identity use

In [ ]:
event={"principal":"agent:old","identity_status":"revoked","action":"read"}
event["identity_status"]=="revoked"

## Lab 17 — Human use of NHI

In [ ]:
event={"actor_type":"human","assumed_identity_type":"workload","break_glass":False}
event["actor_type"]=="human" and event["assumed_identity_type"]=="workload" and not event["break_glass"]

## Lab 18 — Telemetry suppression

In [ ]:
health={"auth_logs":True,"tool_logs":True,"policy_logs":False,"kms_logs":True}
[k for k,v in health.items() if not v]

## Lab 19 — Behavior baseline

In [ ]:
baseline={"claims_api_calls_per_hour":20,"kms_signs_per_hour":2}
observed={"claims_api_calls_per_hour":24,"kms_signs_per_hour":19}
{k:observed[k]/baseline[k] for k in baseline}

## Lab 20 — Rule-based detector

In [ ]:
def detect(e):
    out=[]
    if e.get("identity_status")=="revoked":out.append("revoked_identity_used")
    if e.get("monitoring_disabled"):out.append("telemetry_suppression")
    if e.get("unexpected_audience"):out.append("audience_anomaly")
    return out
detect({"identity_status":"revoked","monitoring_disabled":True})

## Lab 21 — Graph detector: external to crown jewel

In [ ]:
paths=[]
for source,d in G.nodes(data=True):
    if d["kind"]=="external-agent":
        for target in crown_jewels:
            if nx.has_path(G,source,target):paths+=list(nx.all_simple_paths(G,source,target,cutoff=8))
paths

## Lab 22 — Risk-weight an attack path

In [ ]:
edge_risk={"sponsors":5,"delegates":20,"can_access":20,"runs_as":25,"can_sign":40,"federated_access":30,"misconfigured_pivot":50}
def path_risk(path):
    return sum(edge_risk.get(G[path[i]][path[i+1]]["relation"],10) for i in range(len(path)-1))
[(p,path_risk(p)) for p in paths]

## Lab 23 — Posture dimensions

In [ ]:
posture={"inventory":.92,"least_privilege":.61,"credential_hygiene":.73,"delegation":.58,
"detection":.78,"runtime_binding":.88,"external_trust":.70,"response":.69}
weights={"inventory":.10,"least_privilege":.15,"credential_hygiene":.15,"delegation":.15,"detection":.15,"runtime_binding":.10,"external_trust":.10,"response":.10}
round(100*sum(posture[k]*weights[k] for k in posture))

## Lab 24 — Critical posture override

In [ ]:
critical_findings={"leaked_active_credential"}
raw=68
effective=min(raw,25) if critical_findings else raw
effective

## Lab 25 — Detection coverage matrix

In [ ]:
coverage=pd.DataFrame([
{"scenario":"token theft","telemetry":1,"detection":1,"playbook":1},
{"scenario":"delegation abuse","telemetry":1,"detection":1,"playbook":1},
{"scenario":"telemetry suppression","telemetry":1,"detection":1,"playbook":1},
{"scenario":"trust compromise","telemetry":1,"detection":0,"playbook":0}])
coverage

## Lab 26 — Prioritize coverage gaps

In [ ]:
coverage[(coverage.detection==0)|(coverage.playbook==0)]

## Lab 27 — Automated containment plan

In [ ]:
playbooks={"leaked_token":["revoke token","disable renewal","investigate"],
"delegation_abuse":["remove delegation","quarantine child","review parent"],
"workload_compromise":["quarantine workload","revoke sessions","re-attest"]}
playbooks["delegation_abuse"]

## Lab 28 — Scoped kill switch

In [ ]:
kill_switch={"target":"agent:research","block_new_sessions":True,"remove_write":True,
"disable_delegation":True,"preserve_forensics":True}
kill_switch

## Lab 29 — Forensic timeline

In [ ]:
timeline=pd.DataFrame([
{"t":"10:00","event":"token issued","principal":"agent:research"},
{"t":"10:03","event":"unusual token exchange","principal":"agent:research"},
{"t":"10:04","event":"KMS sign attempt","principal":"agent:research"},
{"t":"10:05","event":"quarantined","principal":"agent:research"}])
timeline

## Lab 30 — Persistence hunt

In [ ]:
changes=["new oauth client","new sub-agent","new federation","new api key","normal read"]
[x for x in changes if x.startswith("new ")]

## Lab 31 — Red-team token theft

In [ ]:
attack={"credential":"tok-c","stolen":True,"attempted_resource":"claims-api"}
defense={"token_revoked":True,"renewal_disabled":True,"alert":"IDT-token-leak"}
attack,defense

## Lab 32 — Red-team delegation escalation

In [ ]:
bad={"parent_scope":{"read"},"requested_scope":{"read","write"}}
blocked=not bad["requested_scope"].issubset(bad["parent_scope"])
blocked

## Lab 33 — Red-team signer abuse

In [ ]:
sign_request={"caller":"agent:research","purpose":"arbitrary","payload":"attacker bytes"}
allowed_purposes={"governance-attestation"}
sign_request["purpose"] in allowed_purposes

## Lab 34 — Red-team telemetry evasion

In [ ]:
telemetry_event={"source":"policy-engine","heartbeat_missing":True}
"contain" if telemetry_event["heartbeat_missing"] else "continue"

## Lab 35 — Mean time to contain

In [ ]:
incidents=pd.DataFrame([{"detect":0,"contain":7},{"detect":0,"contain":18},{"detect":0,"contain":4}])
(incidents["contain"]-incidents["detect"]).mean()

## Lab 36 — Attack paths to crown jewels KPI

In [ ]:
sum(1 for s in G.nodes for t in crown_jewels if s!=t and nx.has_path(G,s,t))

## Lab 37 — Remediation by path reduction

In [ ]:
before=len(list(nx.all_simple_paths(G,"external:vendor","kms:claims",cutoff=8)))
G2=G.copy(); G2.remove_edge("external:vendor","agent:research")
after=0 if not nx.has_path(G2,"external:vendor","kms:claims") else len(list(nx.all_simple_paths(G2,"external:vendor","kms:claims",cutoff=8)))
{"before":before,"after":after,"paths_removed":before-after}

## Lab 38 — Identity threat dashboard dataset

In [ ]:
dashboard=pd.DataFrame([
{"metric":"critical attack paths","value":3,"target":0},
{"metric":"leaked active credentials","value":1,"target":0},
{"metric":"high-risk delegations","value":4,"target":0},
{"metric":"detection coverage %","value":86,"target":95},
{"metric":"MTTC minutes","value":9.7,"target":5}])
dashboard

## Lab 39 — Detection-as-code tests

In [ ]:
tests={"revoked_identity":bool(detect({"identity_status":"revoked"})),
"healthy_event_no_finding":not bool(detect({"identity_status":"active"})),
"delegation_escalation":blocked}
tests

## Lab 40 — Response safety

In [ ]:
response={"finding":"suspected trust compromise","automatic":"block new token issuance",
"requires_human":"terminate federation","preserve_evidence":True}
response

## Lab 41 — Recovery validation

In [ ]:
recovery_checks={"persistence_removed":True,"keys_rotated":True,"workload_reattested":True,
"delegations_reviewed":True,"telemetry_healthy":True,"policy_validated":True}
all(recovery_checks.values())

# Capstone — Attack and Defend the Enterprise Agent Identity Graph

Start with a compromised external/research-agent credential. The attacker attempts to:

```text
1. authenticate with the stolen credential
2. obtain a broader token
3. over-delegate to a sub-agent
4. pivot through an MCP/tool service
5. reach the claims API
6. invoke a high-value KMS-backed signer
7. create a persistence identity
8. suppress policy telemetry
```

Your defense must produce machine-verifiable outcomes:

- enumerate the path before the attack;
- score its risk and blast radius;
- detect at least four attack stages;
- block delegation escalation;
- constrain audience/scope;
- quarantine the compromised agent;
- revoke/rotate credentials;
- preserve the forensic timeline;
- remove persistence;
- re-attest before recovery;
- show fewer crown-jewel attack paths after remediation;
- update posture and coverage metrics.

**Success criterion:** do not merely print "blocked." Show which identity relationship or credential was removed, which path disappeared, which evidence was retained, and which control prevented recurrence.


# Review questions

1. Why is an identity graph more useful than a flat entitlement list?
2. How do attack path and blast radius differ?
3. What is a toxic privilege combination?
4. Why should delegation be modeled as its own edge type?
5. How can tools enable lateral movement without shell access?
6. What should be validated beyond a token signature?
7. Why is federation part of the attack surface?
8. Which OWASP NHI risks are especially relevant to agents?
9. How do rule-based and graph-based detections complement each other?
10. What is a critical posture override?
11. Why is telemetry health itself a security signal?
12. How should automated containment be scoped?
13. What must happen before a compromised workload is restored?
14. What evidence is needed to reconstruct delegation abuse?
15. How would you measure whether identity threat defense is improving?
